In [ ]:
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_parquet("hf://datasets/theArijitDas/Fake-Reviews-Dataset/data/train-00000-of-00001.parquet")

df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

x_train_raw, x_test_raw, y_train, y_test = train_test_split(
    df.loc[:, ~df.columns.isin(['label', 'category'])],
    df['label'],
    test_size=.2, random_state=42, shuffle=True
)

tfidf_vectorizer = TfidfVectorizer()
x_train_tfidf = tfidf_vectorizer.fit_transform(x_train_raw['text'])
x_test_tfidf = tfidf_vectorizer.transform(x_test_raw['text'])

x_train_other = csr_matrix(x_train_raw.drop(columns=['text']).values)
x_test_other = csr_matrix(x_test_raw.drop(columns=['text']).values)

x_train_combined = hstack([x_train_tfidf, x_train_other])
x_test_combined = hstack([x_test_tfidf, x_test_other])

In [ ]:
random_forest_rating = sklearn.ensemble.RandomForestClassifier()
param_grid = [{'n_estimators': [100, 300, 1000], 'max_features': ['sqrt', 'log2', .2, .5, 1], 'max_depth': [None, 10, 20, 40], 'min_samples_leaf':[1, 2, 5, 10], 'min_samples_split':[2,5,10]}]

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
rand_search_cv_rating = RandomizedSearchCV(random_forest_rating, param_grid, n_jobs=4, scoring=['accuracy','precision', 'recall', 'f1', 'roc_auc'], refit='f1', verbose=3)
rand_search_cv_rating.fit(x_train_combined, y_train)

y_pred_rating = rand_search_cv_rating.predict(x_test_combined)
f1_refit_rating = f1_score(y_test, y_pred_rating)

print(f"f1 score of RandomForestClassifier w/ RandomizedSearchCV: {f1_refit_rating}\n")
print(f"Best params: {rand_search_cv_rating.best_estimator_}\n")

print(accuracy_score(y_test, y_pred_rating))

## saving best random forest model

In [ ]:
from joblib import dump
dump(rand_search_cv_rating.best_estimator_, 'random_forest_with_rating.joblib')

## XGBoost

In [ ]:
from xgboost import XGBClassifier

xgbclf = XGBClassifier()

param_dist = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [3, 4, 5, 6, 8, 10],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5, 1],
    "min_child_weight": [1, 3, 5, 7, 10],
    "reg_alpha": [0, 0.01, 0.1, 1, 10],
    "reg_lambda": [0.1, 1, 5, 10, 20],
    "scale_pos_weight": [1, 2, 5, 10]
}

In [ ]:
rand_search_cv_xgb = RandomizedSearchCV(xgbclf, param_dist, n_jobs=4, scoring=['accuracy','precision', 'recall', 'f1', 'roc_auc'], refit='f1', verbose=3)
rand_search_cv_xgb.fit(x_train_combined, y_train)

In [ ]:
from sklearn.metrics import recall_score, precision_score
y_pred_xgb = rand_search_cv_xgb.predict(x_test_combined)
print(f"XGB Classifier:\n F1 Score: { f1_score(y_test, y_pred_xgb) }\n Recall: {recall_score(y_test, y_pred_xgb)}\n Precision: {precision_score(y_test, y_pred_xgb)}")

## saving xgb classifier and tfidf vectorizer used

In [ ]:
dump(rand_search_cv_xgb.best_estimator_, 'xgbclf.joblib')
dump(tfidf_vectorizer, 'tfidf_vectorizer.joblib')